# 02 - Solucion de referencia: limpiar y analizar

**Opcional, para despues del ejercicio.** Ejecuta primero el notebook 01 y adjunta
aqui el mismo Lakehouse predeterminado. Este notebook lee las tres tablas creadas,
pero no modifica ni guarda ninguna tabla. Solo muestra resultados en memoria.

Reglas: quitar copias exactas; detenerse si un sale_id tiene versiones distintas;
aceptar ventas completed con quantity > 0 y claves existentes en las dimensiones;
recuperar categoria/ciudad de las dimensiones; calcular net_amount con decimal
y redondeo por linea. Las cancelaciones no se consideran ingresos.


In [ ]:
SCHEMA = "dbo"


In [ ]:
"""Fabric notebook helpers. Uses the existing notebook Spark session."""

import re

TABLE_NAMES = {
    "products": "workshop_products",
    "stores": "workshop_stores",
    "sales_raw": "workshop_sales_raw",
}


def table_names(schema):
    if not isinstance(schema, str) or (
        schema and not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
    ):
        raise ValueError("SCHEMA debe ser un identificador simple o una cadena vacia.")
    prefix = f"`{schema}`." if schema else ""
    return {key: f"{prefix}`{value}`" for key, value in TABLE_NAMES.items()}


def require_default_lakehouse(context):
    if not context.get("defaultLakehouseId"):
        raise RuntimeError(
            "Adjunta un Lakehouse como predeterminado al notebook y reinicia la sesion Spark."
        )


def create_dataframes(spark, data):
    from pyspark.sql import types as T

    schemas = {
        "products": T.StructType(
            [
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("product_name", T.StringType(), False),
                T.StructField("category", T.StringType(), False),
                T.StructField("list_price", T.DecimalType(18, 2), False),
            ]
        ),
        "stores": T.StructType(
            [
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("store_name", T.StringType(), False),
                T.StructField("city", T.StringType(), False),
            ]
        ),
        "sales_raw": T.StructType(
            [
                T.StructField("sale_id", T.StringType(), False),
                T.StructField("sale_date", T.DateType(), False),
                T.StructField("product_id", T.IntegerType(), False),
                T.StructField("store_id", T.IntegerType(), False),
                T.StructField("channel", T.StringType(), False),
                T.StructField("quantity", T.IntegerType(), False),
                T.StructField("unit_price", T.DecimalType(18, 2), False),
                T.StructField("discount_rate", T.DecimalType(4, 2), False),
                T.StructField("status", T.StringType(), False),
                T.StructField("category_raw", T.StringType(), True),
                T.StructField("city_raw", T.StringType(), True),
            ]
        ),
    }
    return {key: spark.createDataFrame(data[key], schema) for key, schema in schemas.items()}


def write_tables(spark, frames, schema, allow_overwrite=False):
    if type(allow_overwrite) is not bool:
        raise ValueError("ALLOW_OVERWRITE debe ser True o False.")
    names = table_names(schema)
    if set(frames) != set(names):
        raise ValueError("Se requieren exactamente products, stores y sales_raw.")
    if schema:
        spark.sql(f"SHOW TABLES IN `{schema}`").limit(1).collect()
    existing = [name for name in names.values() if spark.catalog.tableExists(name)]
    if existing and not allow_overwrite:
        raise FileExistsError(
            "No se ha escrito ninguna tabla. Ya existen: "
            + ", ".join(existing)
            + ". Usa otro Lakehouse o revisa ALLOW_OVERWRITE."
        )
    mode = "overwrite" if allow_overwrite else "errorifexists"
    for key, name in names.items():
        writer = frames[key].write.format("delta").mode(mode)
        if allow_overwrite:
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(name)
    counts = {key: spark.table(name).count() for key, name in names.items()}
    for key, actual in counts.items():
        expected = frames[key].count()
        if actual != expected:
            raise RuntimeError(f"Recuento incorrecto en {names[key]}: {actual} != {expected}")
    return counts


In [ ]:
"""Reference solution: exact deduplication, canonical dimensions and decimal revenue."""


def clean_sales(sales_raw, products, stores):
    from pyspark.sql import functions as F

    for frame, key in ((products, "product_id"), (stores, "store_id")):
        if frame.filter(F.col(key).isNull()).limit(1).count():
            raise ValueError(f"La dimension contiene una clave nula: {key}")
        if frame.groupBy(key).count().filter(F.col("count") > 1).limit(1).count():
            raise ValueError(f"La dimension contiene claves duplicadas: {key}")

    deduplicated = sales_raw.dropDuplicates()
    if deduplicated.filter(F.col("sale_id").isNull()).limit(1).count():
        raise ValueError("Hay sale_id nulos. Revisa los datos antes de continuar.")
    if deduplicated.groupBy("sale_id").count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError("Hay sale_id repetidos con contenido diferente. No se elige uno al azar.")

    eligible = deduplicated.filter((F.col("quantity") > 0) & (F.col("status") == "completed"))
    enriched = eligible.join(products.select("product_id", "category"), "product_id", "inner").join(
        stores.select("store_id", "city"), "store_id", "inner"
    )
    return enriched.select(
        "sale_id",
        "sale_date",
        "product_id",
        "store_id",
        "channel",
        "quantity",
        "unit_price",
        "discount_rate",
        "status",
        "category",
        "city",
        F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1) - F.col("discount_rate")), 2)
        .cast("decimal(18,2)")
        .alias("net_amount"),
    )


In [ ]:
import notebookutils
from pyspark.sql import functions as F

require_default_lakehouse(notebookutils.runtime.context)
names = table_names(SCHEMA)
source_frames = {key: spark.table(name) for key, name in names.items()}
sales_clean = clean_sales(
    source_frames["sales_raw"], source_frames["products"], source_frames["stores"]
)
summary = sales_clean.groupBy("category").agg(
    F.count("*").alias("sales_count"),
    F.sum("quantity").alias("units"),
    F.sum("net_amount").alias("net_revenue"),
)
display(summary.orderBy(F.desc("net_revenue"), "category"))
display(sales_clean.agg(
    F.count("*").alias("clean_rows"),
    F.sum("net_amount").alias("net_revenue"),
))
assert sales_clean.count() == sales_clean.select("sale_id").distinct().count()
assert sales_clean.filter(
    F.col("category").isNull() | F.col("city").isNull()
).count() == 0


## Comprobar y explicar

Compara recuentos e ingresos con el informe del notebook 01. Con su configuracion
original tambien puedes consultar `docs/expected-results.json` en GitHub.
Si cambias la semilla o el volumen, los importes de ese archivo ya no aplican.
El detalle de negocio se explica en `docs/datos.md`.

Para persistir una tabla limpia, pide a Copilot un plan que indique destino, esquema,
comprobaciones y modo de escritura. Este notebook no realiza esa escritura.
